In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import HTML, Markdown
import xarray as xr
import plotly.graph_objects as go
import seaborn as sns

import nn_gmm as nng
import ml_tools as mlt
import seismic_hazard_analysis as sha
import oq_wrapper as oqw

device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
if torch.mps.is_available():
    device = "mps"

print(f"Using device: {device.upper()}")

In [ ]:
site = "NPCS"
model_dir = Path("/Users/claudy/dev/work/data/nn_gmm/results/1230_1645_full_v1_3Epochs")

In [ ]:
run_config = nng.nn_gmm.load_config(model_dir / "run_config.yaml")

# Load the ERF file
background_ffp = (
    nng.constants.HAZARD_RESOURCES_DIR
    / "NZBCK211_OpenSHA.txt"
)
ds_erf_ffp = nng.constants.HAZARD_RESOURCES_DIR / "NZ_DSmodel_2010.txt"

ds_erf_df = pd.read_csv(ds_erf_ffp, index_col="rupture_name")
ds_source_df = sha.nshm_2010.get_ds_source_df(background_ffp)

In [ ]:
active_shallow_ids = ds_source_df.loc[ds_source_df["tectonic_type"] == "ACTIVE_SHALLOW"].index.values.astype(str)
subduction_ids = ds_source_df.loc[ds_source_df["tectonic_type"] == "SUBDUCTION_SLAB"].index.values.astype(str)

print("Number of ACTIVE_SHALLOW sources:", len(active_shallow_ids))
print("Number of SUBDUCTION_SLAB sources:", len(subduction_ids))

In [ ]:
with nng.DuckIMDB(run_config.imdb_ffp, readonly=True) as imdb:
    site_df = imdb.get_site_df(add_nztm=True).set_index("site_id")

In [ ]:
# Get GM parameters using empirical GMM
GMM_MAPPING = {
    oqw.constants.TectType.ACTIVE_SHALLOW: oqw.constants.GMMLogicTree.NSHM2022,
    oqw.constants.TectType.SUBDUCTION_SLAB: oqw.constants.GMMLogicTree.NSHM2022,
    oqw.constants.TectType.SUBDUCTION_INTERFACE: oqw.constants.GMMLogicTree.NSHM2022,
}

site_nztm = site_df.loc[site, ["nztm_x", "nztm_y"]].values
site_properties = {
    "vs30": site_df.loc[site, "vs30"],
    "z1p0": site_df.loc[site, "z1p0"],
    "z2p5": site_df.loc[site, "z2p5"],
    "backarc": False,
    "vs30measured": True,
}

emp_rupture_df = sha.nshm_2010.get_oq_ds_rupture_df(
    ds_source_df, site_nztm, site_properties
)

emp_rupture_df = emp_rupture_df.loc[emp_rupture_df.rrup <= 300]
emp_gm_params = sha.nshm_2010.get_emp_gm_params(emp_rupture_df, GMM_MAPPING, run_config.ims)

In [ ]:
emp_mean_df = emp_gm_params[[f"{im}_mean" for im in run_config.ims]].rename(
    columns=dict(zip([f"{im}_mean" for im in run_config.ims], run_config.ims))
).sort_index()

In [ ]:
nn_rupture_df = ds_source_df.copy()
site_series = site_df.loc[site] 

# Compute site distances
nn_rupture_df["rjb"] = (
    np.sqrt(
        (site_series.loc["nztm_x"] - nn_rupture_df["nztm_x"]) ** 2
        + (site_series.loc["nztm_y"] - nn_rupture_df["nztm_y"]) ** 2
    )
    / 1000
)
nn_rupture_df["rrup"] = (
    np.sqrt(
        (site_series.loc["nztm_x"] - nn_rupture_df["nztm_x"]) ** 2
        + (site_series.loc["nztm_y"] - nn_rupture_df["nztm_y"]) ** 2
        + (nn_rupture_df["depth"] * 1000) ** 2
    )
    / 1000
)
# Use Rjb for rx and ry, in the past we have used zero for this.
# Using Rjb gives the same result (when using Br13 and ZA06)
# as using zero, and makes more sense.
nn_rupture_df["rx"] = nn_rupture_df["rjb"]
nn_rupture_df["ry"] = nn_rupture_df["rjb"]

# Apply max-rrup limit
nn_rupture_df = nn_rupture_df.loc[nn_rupture_df.rrup <= run_config.max_rrup]

# Add site properties
nn_rupture_df["vs30"] = site_series.loc["vs30"]
nn_rupture_df["z1p0"] = site_series.loc["z1p0"]
nn_rupture_df["z2p5"] = site_series.loc["z2p5"]


# Rename columns to match expected input names
nn_rupture_df = nn_rupture_df.rename(
    columns={"mag": "magnitude", "tectonic_type": "tect_type", "dbot": "dbottom", "depth": "hypo_depth"}
)
nn_rupture_df["is_point_source"] = True

assert np.all(
    np.isin(run_config.source_inputs, nn_rupture_df.columns)
), "Source inputs missing"
assert np.all(
    np.isin(run_config.site_inputs, nn_rupture_df.columns)
), "Site inputs missing"
assert np.all(
    np.isin(run_config.source_to_site_inputs, nn_rupture_df.columns)
), "Source-to-site inputs missing"

# Convert tectonic type
nn_rupture_df["tect_type"] = nn_rupture_df["tect_type"].cat.rename_categories({
    "SUBDUCTION_SLAB": "SUBDUCTION_INTERFACE",
})


pred_df = nng.nn_gmm.run_predictions_dir(model_dir, nn_rupture_df, device)

In [ ]:
pred_mean_cols = [f"{im}_pred" for im in run_config.ims]
pred_mean_df = pred_df[pred_mean_cols].rename(columns=dict(zip(pred_mean_cols, run_config.ims))).sort_index()

In [ ]:
assert pred_mean_df.index.equals(emp_mean_df.index)

In [ ]:
active_shallow_ids = active_shallow_ids[mlt.array_utils.pandas_isin(active_shallow_ids, pred_mean_df.index)]
subduction_ids = subduction_ids[mlt.array_utils.pandas_isin(subduction_ids, pred_mean_df.index)]

print("Number of ACTIVE_SHALLOW sources with predictions:", len(active_shallow_ids))
print("Number of SUBDUCTION_SLAB sources with predictions:", len(subduction_ids))

## Comparison of mean values

In [ ]:
ims = nng.constants.PLOT_IMS
fig, axs = mlt.plotting.get_fig_axes(len(ims), 2, -1, (8, 6))

for i, (im, ax) in enumerate(zip(ims, axs)):
    # NN all
    nn_im_values = np.sort(pred_mean_df[im].values)
    weights = np.arange(1, len(nn_im_values) + 1) / len(nn_im_values)
    ax.step(nn_im_values, weights, label="NN-GMM", color="blue")

    # NN active_shallow
    nn_active_shallow_values = np.sort(
        pred_mean_df.loc[active_shallow_ids, im].values
    )
    weights = np.arange(1, len(nn_active_shallow_values) + 1) / len(nn_active_shallow_values)
    ax.step(
        nn_active_shallow_values,
        weights,
        label="NN-GMM ACTIVE_SHALLOW",
        color="blue",
        linestyle="dashed",
    )

    # NN subduction_slab
    nn_subduction_values = np.sort(
        pred_mean_df.loc[subduction_ids, im].values
    )
    weights = np.arange(1, len(nn_subduction_values) + 1) / len(nn_subduction_values)
    ax.step(
        nn_subduction_values,
        weights,
        label="NN-GMM SUBDUCTION_SLAB",
        color="blue",
        linestyle="dotted",
    )

    # Empirical GMM
    emp_im_values = np.sort(emp_mean_df[im].values)
    weights = np.arange(1, len(emp_im_values) + 1) / len(emp_im_values)
    ax.step(emp_im_values, weights, label="Empirical GMM", color="orange")

    # Empirical GMM active_shallow
    emp_active_shallow_values = np.sort(
        emp_mean_df.loc[active_shallow_ids, im].values
    )
    weights = np.arange(1, len(emp_active_shallow_values) + 1) / len(emp_active_shallow_values)
    ax.step(
        emp_active_shallow_values,
        weights,
        label="Empirical GMM ACTIVE_SHALLOW",
        color="orange",
        linestyle="dashed",
    )

    # Empirical GMM subduction_slab
    emp_subduction_values = np.sort(
        emp_mean_df.loc[subduction_ids, im].values
    )
    weights = np.arange(1, len(emp_subduction_values) + 1) / len(emp_subduction_values)
    ax.step(
        emp_subduction_values,
        weights,
        label="Empirical GMM SUBDUCTION_SLAB",
        color="orange",
        linestyle="dotted",
    )

    if i == 0:
        ax.legend()

    ax.grid(linewidth=0.5, alpha=0.5, linestyle="--")
    ax.set_ylim(0, 1)
    ax.set_xlabel(f"{im} (g)")
    ax.set_ylabel("Density")

fig.tight_layout()

## Subduction Events

### Magnitude Histogram

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 6))

ax.hist(
    ds_source_df.loc[subduction_ids, "mag"],
    bins=25,
    alpha=0.7,
    edgecolor="k",
)
ax.set_xlabel("Magnitude")
ax.set_ylabel("Number of Ruptures")

### Weighted Magnitude Histogram

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(16, 6))

ax.hist(
    ds_source_df.loc[subduction_ids, "mag"],
    weights=ds_erf_df.loc[subduction_ids, "annual_rec_prob"],
    bins=25,
    alpha=0.7,
    edgecolor="k",
)
ax.set_xlabel("Magnitude")
ax.set_ylabel("Weighted (Annual Recurrence) Number of Ruptures")